In [6]:
import sys
from pathlib import Path

# Set root
cwd = Path.cwd()
ROOT = next(p for p in [cwd, *cwd.parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src.features.preprocessing import load_model_data
from src.evaluation import splits
from src.evaluation.make_splits import load_split

In [7]:
# Load the cleaned/merged dataset

X, y, meta = load_model_data()
FEATURES = list(X.columns)

data = load_split(pd.concat([meta, X], axis=1).assign(log_nitrate=y))
print(data["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(data["holdout"].sum()))

# df = pd.read_csv(DATA_PATH, dtype={"gm_well_id": "string"}, parse_dates=["gm_samp_collection_date"])
# df = load_split(df)

# print(df["fold"].value_counts().sort_index().to_dict(), "| holdout rows:", int(df["holdout"].sum()))

{0: 1035, 1: 972, 2: 1143, 3: 1041, 4: 1089} | holdout rows: 1089


In [8]:
# Set the training folds and the holdout

train_val = data[~data["holdout"]].reset_index(drop=True)       # Training set with 4 folds


X_tv = train_val[FEATURES]
y_tv = train_val["log_nitrate"]

# Print the sizes of the training folds
print(train_val["fold"].value_counts().sort_index().to_dict())

{0: 1035, 1: 972, 2: 1143, 3: 1041}


Here you can begin training your model.

In [9]:
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error

In [10]:
def make_model():
    """Replace with your model. A new, untrained model is created for every fold."""
    return DummyRegressor(strategy="mean")        # baseline: always predicts the training mean

oof_pred = np.full(len(train_val), np.nan)        # out-of-fold prediction for every row
fold_scores = []

for train, test in splits.iter_folds(train_val["fold"]):
    model = make_model()
    model.fit(X_tv[train], y_tv[train])
    oof_pred[test] = model.predict(X_tv[test])

    fold = int(train_val.loc[test, "fold"].iloc[0])
    fold_scores.append({"fold": fold,
                        "rmse_log": root_mean_squared_error(y_tv[test], oof_pred[test])})

fold_scores = pd.DataFrame(fold_scores)
exceeds = np.expm1(y_tv) > 10
print(fold_scores.round(3).to_string(index=False))
print(f"Overall RMSE (log): {root_mean_squared_error(y_tv, oof_pred):.3f}")
print(f"Accuracy of the >10 mg/L call: {np.mean((np.expm1(oof_pred) > 10) == exceeds):.3f}")


 fold  rmse_log
    0     1.227
    1     1.183
    2     1.222
    3     1.314
Overall RMSE (log): 1.238
Accuracy of the >10 mg/L call: 0.662


Save this last file for the final evaluation (when we have compared all of our models and have chosen a final one).

In [11]:
# ============================================================
# FINAL EVALUATION ON THE HOLD-OUT FOLD
# Run this ONCE, for the final chosen model only, after all
# model choices (features, hyperparameters) are finished.
# ============================================================
RUN_FINAL_EVALUATION = False

if RUN_FINAL_EVALUATION:
    holdout_df = df[df["holdout"]].reset_index(drop=True)
    final_model = ...                      # the chosen model, refit on all of train_val
    final_model.fit(X, y)
    holdout_pred = final_model.predict(holdout_df[FEATURES])
    ...                                    # scores on np.log1p(holdout_df[TARGET])
